# posek_processed.kubikov vs heatmap_past_data.target

Are these the same data? `kubikov` is actual harvested m³; `target` is the model's predicted/expected activity score. This notebook joins them on `(odsek, ggo, month)` and compares the values directly.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("..").resolve()

posek   = pd.read_csv(BASE / "data" / "posek_processed.csv")
heatmap = pd.read_csv(BASE / "data" / "heatmap_past_data.csv")

posek.columns   = posek.columns.str.strip()
heatmap.columns = heatmap.columns.str.strip()

print(f"posek_processed:  {len(posek):>8,} rows")
print(f"heatmap_past:     {len(heatmap):>8,} rows")

In [ ]:
# ── Normalise posek ──────────────────────────────────────────────────────────
# ggo: zero-padded string "01" → integer
posek["ggo_int"] = posek["ggo"].astype(str).str.strip().str.lstrip('0').replace('', '0').astype(int)
posek["odsek"]   = posek["odsek"].astype(str).str.strip().str.replace(' ', '', regex=False)
posek["kubikov"] = pd.to_numeric(posek["kubikov"], errors="coerce").fillna(0)

# Build leto_mesec key
posek["leto_mesec"] = posek["leto"].astype(int).astype(str) + "-" + posek["mesec"].astype(int).apply(lambda m: f"{m:02d}")

# Aggregate: sum kubikov per (odsek, ggo, month) — multiple vrsec rows collapse into one
posek_agg = (
    posek.groupby(["odsek", "ggo_int", "leto_mesec"], as_index=False)["kubikov"]
    .sum()
    .rename(columns={"odsek": "odsek_id", "ggo_int": "ggo"})
)

# ── Normalise heatmap ─────────────────────────────────────────────────────────
heatmap["odsek_id"]   = heatmap["odsek_id"].astype(str).str.strip().str.replace(' ', '', regex=False)
heatmap["ggo"]        = pd.to_numeric(heatmap["ggo"], errors="coerce").astype("Int64")
heatmap["target"]     = pd.to_numeric(heatmap["target"], errors="coerce").fillna(0)
heatmap["leto_mesec"] = heatmap["leto_mesec"].astype(str).str.strip()

# Keep only one row per (odsek, ggo, month) — heatmap may have duplicates across GGO
# (server takes max; we do the same here)
heatmap_agg = (
    heatmap.groupby(["odsek_id", "ggo", "leto_mesec"], as_index=False)["target"]
    .max()
)

print(f"posek after aggregation:   {len(posek_agg):>8,} rows")
print(f"heatmap after aggregation: {len(heatmap_agg):>8,} rows")

In [ ]:
# ── Inner join on (odsek_id, ggo, leto_mesec) ────────────────────────────────
merged = pd.merge(
    posek_agg,
    heatmap_agg,
    on=["odsek_id", "ggo", "leto_mesec"],
    how="inner"
)

only_posek   = len(posek_agg)   - len(merged)
only_heatmap = len(heatmap_agg) - len(merged)

print(f"Rows in both (inner join):      {len(merged):>8,}")
print(f"Only in posek (no heatmap row): {only_posek:>8,}")
print(f"Only in heatmap (no posek row): {only_heatmap:>8,}")
print()
print(merged[["odsek_id","ggo","leto_mesec","kubikov","target"]].head(10).to_string(index=False))

In [ ]:
# ── Are the values the same? ─────────────────────────────────────────────────
merged["diff"]     = (merged["kubikov"] - merged["target"]).abs()
merged["rel_diff"] = merged["diff"] / merged["target"].replace(0, np.nan)

# Exact match (rounded to 2 dp to ignore floating point noise)
exact = (merged["kubikov"].round(2) == merged["target"].round(2)).sum()
close = (merged["diff"] < 0.01).sum()   # within 1 cent / 0.01 m³

print(f"Exact match (2 dp):         {exact:>8,} / {len(merged):,}  ({exact/len(merged)*100:.1f}%)")
print(f"Difference < 0.01:          {close:>8,} / {len(merged):,}  ({close/len(merged)*100:.1f}%)")
print()
print("Absolute difference — distribution:")
print(merged["diff"].describe().round(4))
print()
print("Relative difference (where target > 0) — distribution:")
print(merged["rel_diff"].dropna().describe().round(4))

In [ ]:
# ── Rows where values differ substantially ────────────────────────────────────
BIG_DIFF_THRESHOLD = 1.0   # more than 1 m³ absolute difference

big_diff = merged[merged["diff"] > BIG_DIFF_THRESHOLD].sort_values("diff", ascending=False)
print(f"Rows with |kubikov - target| > {BIG_DIFF_THRESHOLD}: {len(big_diff):,}")
print()
if len(big_diff):
    print(big_diff[["odsek_id","ggo","leto_mesec","kubikov","target","diff"]].head(20).to_string(index=False))

In [ ]:
# ── Cases where one is zero and the other is not ─────────────────────────────
posek_zero_target_not = merged[(merged["kubikov"] == 0) & (merged["target"] > 0)]
target_zero_posek_not = merged[(merged["target"] == 0) & (merged["kubikov"] > 0)]

print(f"kubikov=0 but target>0: {len(posek_zero_target_not):,}  (heatmap shows activity, posek has nothing)")
print(f"target=0 but kubikov>0: {len(target_zero_posek_not):,}  (posek recorded m³, heatmap shows zero)")
print()
if len(target_zero_posek_not):
    print("Sample where target=0 but kubikov>0:")
    print(target_zero_posek_not[["odsek_id","ggo","leto_mesec","kubikov","target"]].head(10).to_string(index=False))

In [ ]:
# ── Summary verdict ───────────────────────────────────────────────────────────
print("=" * 60)
print("VERDICT")
print("=" * 60)

same = exact == len(merged)
near_same = close / len(merged) > 0.99

if same:
    print("\nValues are IDENTICAL (to 2 decimal places) — same data source.")
elif near_same:
    print("\nValues are NEARLY IDENTICAL (>99% within 0.01) — likely same")
    print("source with minor floating point differences.")
else:
    max_diff = merged["diff"].max()
    mean_diff = merged["diff"].mean()
    print(f"\nValues DIFFER substantially.")
    print(f"  Mean absolute diff: {mean_diff:.4f}")
    print(f"  Max absolute diff:  {max_diff:.4f}")
    print(f"  Exact matches (2dp): {exact/len(merged)*100:.1f}%")
    print("\nConclusion: kubikov and target are NOT the same data.")
    print("posek_processed cannot be replaced by heatmap_past_data.")